In [ ]:
# ============================================================
#  环境配置
#  - Colab 用户：取消注释下方 Colab 区块
#  - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch torchvision -q
# !pip install matplotlib numpy -q

# ── 本地 Jupyter 环境 ──
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_install("torch")
_install("torchvision")
_install("matplotlib")
_install("numpy")

In [ ]:
import math
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
from torchvision.models.video import r2plus1d_18

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# R(2+1)D 代码实战：学习实现 vs 工程实现

基于论文 *A Closer Look at Spatiotemporal Convolutions for Action Recognition*（Tran et al., CVPR 2018），
用 **视频分类** 任务演示 R(2+1)D 的核心思想：把 3D 卷积拆成 **空间 2D 卷积 + 时间 1D 卷积**。

本 Notebook 使用 **同一份 toy 视频数据**，并行展示两条路径：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解 R(2+1)D 为什么比 R3D 更容易训练 | 掌握工业库中如何快速搭建视频模型 |
| 实现方式 | 手写 `(2+1)D` block + 手写小型网络 + 与 R3D 对照训练 | `torchvision.models.video.r2plus1d_18` builder 直接搭建 |
| 代码量 | ~120 行 | ~20 行 |
| 适合场景 | 面试准备、原理拆解、研究原型 | 工程验证、复用成熟实现、快速试错 |

> 两条路径不是两套无关代码，而是同一时空建模思想在不同抽象层级上的表达。

## 一、论文背景、任务与范围

**论文**：*A Closer Look at Spatiotemporal Convolutions for Action Recognition*  
**作者**：Du Tran, Heng Wang, Lorenzo Torresani, Jamie Ray, Yann LeCun, Manohar Paluri  
**会议**：CVPR 2018

### 任务定义
R(2+1)D 面向的是 **视频动作识别 / 视频分类**。输入不是单张图像，而是一段 clip，模型需要同时建模：
- **空间信息**：每一帧里“看到了什么”
- **时间信息**：帧与帧之间“如何变化”

### 为什么这个 toy 任务足够说明问题
这里不用 Kinetics 或 Sports-1M，而是用一个可直接运行的合成视频数据集：
- 水平条纹下移
- 垂直条纹右移
- 中心方块逐步扩张

这个任务虽然简单，但已经具备了 R(2+1)D 最关心的两件事：**空间模式** 和 **时间演化**。

### 架构谱系与本 Notebook 的边界
可把这条线索理解成：**2D CNN → C3D / I3D / R3D → R(2+1)D → SlowFast / X3D**。

本 Notebook 聚焦：
1. 为什么把 3D 卷积拆开后更容易优化
2. `M_i` 参数量匹配公式如何落到代码
3. 手写实现与工业库实现的对应关系

本 Notebook **不覆盖**：大规模真实视频训练、预训练权重迁移实验、复杂视频增强流水线。

## 二、最小必要理论与可行性决策

### 1. 论文核心公式
标准 3D 卷积直接联合建模时间和空间：

$$y = \text{Conv3D}(x;\; t \times d \times d)$$

R(2+1)D 把它拆成两步：

$$x' = \text{Conv}_{space}(x;\; 1 \times d \times d)$$
$$x'' = \text{ReLU}(x')$$
$$y = \text{Conv}_{time}(x'';\; t \times 1 \times 1)$$

如果希望分解前后参数量尽量接近，中间通道数取：

$$M_i = \left\lfloor \frac{t d^2 N_{i-1} N_i}{d^2 N_{i-1} + t N_i} \right\rfloor$$

直观理解：
- **先学空间，再学时间**，优化目标更清晰
- 中间多了一次 **ReLU**，表达能力更强
- 在参数量大致不变时，论文报告 R(2+1)D 比 R3D 更容易优化

### 2. 本 Notebook 的四个决策
```yaml
model_name: "R(2+1)D"
paper: "A Closer Look at Spatiotemporal Convolutions for Action Recognition (Tran et al., 2018)"
task_type: "video_classification"
learning_path_depth: "L1"
learning_rationale: "toy 视频分类可稳定完成从零训练，并能直接对比 R3D vs R(2+1)D"
engineering_path_form: "E1"
engineering_rationale: "torchvision 提供成熟的 r2plus1d_18 builder"
engineering_action: "train"
action_rationale: "为了保持同一任务/同一数据、避免外网依赖，并保证 CPU/GPU 都能跑通，这里使用 builder 从头搭建并训练小规模示例"
training_vs_inference_diff: true
```

> 这里把 `training_vs_inference_diff` 设为 `true`，但差异不是 Transformer 那种 teacher forcing，而是 **BatchNorm / 梯度 / `model.train()` vs `model.eval()`** 的运行模式差别。

## 三、论文组件映射表

| 论文组件 | 学习路径覆盖 | 工程库/API 对应 | 状态 |
|---------|-------------|----------------|------|
| 五种时空卷积架构（R2D / MCx / rMCx / R3D / R(2+1)D） | 先用 markdown 交代设计空间，再重点实现 R3D 与 R(2+1)D | `torchvision` 直接提供 `r3d_18` / `r2plus1d_18`，其余结构本 Notebook 仅解释 | Explain-only |
| 空间 2D 卷积 `1 × d × d` | `R2Plus1DConv.spatial` 手写实现 | `r2plus1d_18` 内部 `Conv2Plus1D` 的 spatial part | Runnable |
| 中间 ReLU 非线性 | 显式插入在 spatial 与 temporal 之间 | builder 内部激活层 | Runnable |
| 时间 1D 卷积 `t × 1 × 1` | `R2Plus1DConv.temporal` 手写实现 | `Conv2Plus1D` 的 temporal part | Runnable |
| 参数量匹配公式 `M_i` | 显式计算并打印与 R3D 的参数比 | 工程库内部隐藏，不需要手动推导 | Runnable |
| R3D 基线 | `R3DConv` / `R3DNet` 从零实现并训练 | `torchvision.models.video.r3d_18` 属于同家族 builder | Runnable |
| “R(2+1)D 更易优化”结论 | 用 toy 任务中的 loss 曲线做小规模验证 | 工程路径只展示成熟实现，不再复刻完整论文消融 | Illustrative |

## 四、数据准备

为了满足 **Colab 免费环境 / CPU 可运行 / 无需手动下载** 这几个约束，这里构造一个小型 synthetic video 数据集。

每个样本都是形状 `(C=1, T=8, H=16, W=16)` 的灰度视频，类别共有 3 类：
1. 水平条纹向下移动
2. 垂直条纹向右移动
3. 中心方块逐步扩张

这样做的好处是：
- 不依赖外部数据下载
- 训练稳定，几分钟内可跑完
- 依然保留了 **空间模式 + 时间变化** 这两个动作识别的核心难点

> 工程路径会复用同一任务，但由于 `torchvision` 的 `r2plus1d_18` 期望 3 通道输入，我们会把灰度 clip 在通道维复制为 RGB。

In [ ]:
# ── Hyperparameters（两条路径共用） ──
NUM_CLASSES = 3
N_FRAMES = 8
IMG_H, IMG_W = 16, 16
T_KERNEL = 3
S_KERNEL = 3

TRAIN_SAMPLES = 480
TEST_SAMPLES = 120

LEARNING_BATCH_SIZE = 32
LEARNING_EPOCHS = 12
LEARNING_LR = 1e-3

ENGINEERING_TRAIN_SAMPLES = 64   # builder 路径只做快速演示，控制 CPU 耗时
ENGINEERING_BATCH_SIZE = 8
ENGINEERING_EPOCHS = 2
ENGINEERING_LR = 5e-4


class SyntheticVideoDataset(Dataset):
    """三类合成视频：每类体现不同的时空变化模式。"""
    def __init__(self, n_samples=480):
        self.videos, self.labels = [], []
        for i in range(n_samples):
            label = i % NUM_CLASSES
            video = torch.zeros(1, N_FRAMES, IMG_H, IMG_W)                 # (1, T, H, W)
            for t in range(N_FRAMES):
                if label == 0:  # horizontal stripe moves down
                    row = (2 + t) % IMG_H
                    video[0, t, row:min(row + 3, IMG_H), :] = 1.0
                elif label == 1:  # vertical stripe moves right
                    col = (2 + t) % IMG_W
                    video[0, t, :, col:min(col + 3, IMG_W)] = 1.0
                else:  # center square grows over time
                    s = 1 + t // 2
                    cy, cx = IMG_H // 2, IMG_W // 2
                    video[0, t, cy - s:cy + s, cx - s:cx + s] = 1.0

            video += 0.05 * torch.randn_like(video)
            self.videos.append(video)
            self.labels.append(label)

        self.videos = torch.stack(self.videos)                            # (N, 1, T, H, W)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.videos[idx], self.labels[idx]


train_ds = SyntheticVideoDataset(TRAIN_SAMPLES)
test_ds = SyntheticVideoDataset(TEST_SAMPLES)

learning_train_loader = DataLoader(train_ds, batch_size=LEARNING_BATCH_SIZE, shuffle=True)
engineering_train_loader = DataLoader(
    Subset(train_ds, range(ENGINEERING_TRAIN_SAMPLES)),
    batch_size=ENGINEERING_BATCH_SIZE,
    shuffle=True,
)
test_loader = DataLoader(test_ds, batch_size=LEARNING_BATCH_SIZE, shuffle=False)

sample_videos, sample_labels = next(iter(learning_train_loader))
print(f'Learning batch: {tuple(sample_videos.shape)}')   # (B, 1, T, H, W)
print(f'Labels shape  : {tuple(sample_labels.shape)}')

frame_ids = [0, 2, 4, 6]
fig, axes = plt.subplots(1, len(frame_ids), figsize=(10, 2.5))
video0 = sample_videos[0]
label0 = sample_labels[0].item()
for ax, frame_id in zip(axes, frame_ids):
    ax.imshow(video0[0, frame_id].numpy(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Frame {frame_id}')
    ax.axis('off')
fig.suptitle(f'Synthetic Video Example | label={label0}')
plt.tight_layout()
plt.show()

In [ ]:
# ── Shared utilities（训练 / 评估 / 适配） ──
def adapt_to_rgb(videos):
    """(B, 1, T, H, W) -> (B, 3, T, H, W) for torchvision video models."""
    return videos.repeat(1, 3, 1, 1, 1)


def train_model(model, loader, epochs, lr, input_adapter=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    model = model.to(device)
    losses = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for videos, labels in loader:
            if input_adapter is not None:
                videos = input_adapter(videos)

            videos = videos.to(device)
            labels = labels.to(device)

            logits = model(videos)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)
        print(f'Epoch [{epoch + 1:02d}/{epochs}] | loss = {avg_loss:.4f}')

    return losses


def evaluate_model(model, loader, input_adapter=None):
    model = model.to(device)
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for videos, labels in loader:
            if input_adapter is not None:
                videos = input_adapter(videos)

            videos = videos.to(device)
            labels = labels.to(device)
            preds = model(videos).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


def predict_batch(model, videos, input_adapter=None):
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        if input_adapter is not None:
            videos = input_adapter(videos)
        logits = model(videos.to(device))
        return logits.cpu(), logits.argmax(dim=1).cpu()


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 五、学习路径：从零实现 R(2+1)D

### 1. 先看论文里的五种结构

| 架构 | 核心想法 | 本 Notebook 是否实现 |
|------|----------|----------------------|
| R2D | 把时间维拼到通道，退化成纯 2D CNN | 否，仅解释 |
| MCx | 底层 3D，上层 2D | 否，仅解释 |
| rMCx | 底层 2D，上层 3D | 否，仅解释 |
| R3D | 全 3D 卷积 | 是，作为对照组 |
| R(2+1)D | 3D 卷积拆成空间 2D + 时间 1D | 是，重点实现 |

### 2. 关键模块的数据流
对输入 clip `x ∈ R^{B×C×T×H×W}`，一个 `(2+1)D` block 的形状变化是：

$$
(B, C, T, H, W)
\xrightarrow{1 \times d \times d}
(B, M, T, H, W)
\xrightarrow{ReLU}
(B, M, T, H, W)
\xrightarrow{t \times 1 \times 1}
(B, N, T, H, W)
$$

这里的重点不是“参数更少”，而是：
- **空间模式** 和 **时间模式** 被拆开学习
- 中间插入额外非线性后，表达能力更强
- 在参数量相近时，更容易优化

下面先把单个 block 写出来，再组装成小型分类网络。

In [ ]:
class R2Plus1DConv(nn.Module):
    """Spatial 2D conv -> ReLU -> Temporal 1D conv."""
    def __init__(self, in_ch, out_ch, t=T_KERNEL, d=S_KERNEL):
        super().__init__()
        self.mid_ch = max(
            1,
            math.floor((t * d * d * in_ch * out_ch) / (d * d * in_ch + t * out_ch))
        )

        self.spatial = nn.Sequential(
            nn.Conv3d(in_ch, self.mid_ch, kernel_size=(1, d, d), padding=(0, d // 2, d // 2)),
            nn.BatchNorm3d(self.mid_ch),
            nn.ReLU(inplace=True),
        )
        self.temporal = nn.Sequential(
            nn.Conv3d(self.mid_ch, out_ch, kernel_size=(t, 1, 1), padding=(t // 2, 0, 0)),
            nn.BatchNorm3d(out_ch),
        )

    def forward(self, x):
        x = self.spatial(x)    # (B, in_ch, T, H, W) -> (B, mid_ch, T, H, W)
        x = self.temporal(x)   # (B, mid_ch, T, H, W) -> (B, out_ch, T, H, W)
        return x


class R3DConv(nn.Module):
    """Standard 3D convolution block for baseline comparison."""
    def __init__(self, in_ch, out_ch, t=T_KERNEL, d=S_KERNEL):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=(t, d, d), padding=(t // 2, d // 2, d // 2)),
            nn.BatchNorm3d(out_ch),
        )

    def forward(self, x):
        return self.block(x)   # (B, in_ch, T, H, W) -> (B, out_ch, T, H, W)

In [ ]:
# ── 参数量验证：R3D vs R(2+1)D ──
configs = [(1, 16), (16, 32), (32, 64)]
print(f'{"Layer":>10} | {"R3D params":>12} | {"R(2+1)D params":>14} | {"M_i":>4} | {"Ratio":>6}')
print('-' * 66)
for in_ch, out_ch in configs:
    r3d = R3DConv(in_ch, out_ch)
    r21d = R2Plus1DConv(in_ch, out_ch)
    p3d = count_params(r3d)
    p21d = count_params(r21d)
    print(f'{in_ch:>4} -> {out_ch:<4} | {p3d:>12,} | {p21d:>14,} | {r21d.mid_ch:>4} | {p21d / p3d:>5.2f}x')

dummy = torch.randn(2, 1, N_FRAMES, IMG_H, IMG_W)
out_r3d = R3DConv(1, 16)(dummy)
out_r21d = R2Plus1DConv(1, 16)(dummy)
print(f'R3D output shape     : {tuple(out_r3d.shape)}')
print(f'R(2+1)D output shape : {tuple(out_r21d.shape)}')

## 六、把 block 组装成完整网络

学习路径这里采用 **L1: full mini training**。

也就是说，我们不仅验证单个 block 形状正确，还真正训练一个小型 R(2+1)D 分类器，并与小型 R3D 对照组比较。这样最能对应论文“在参数量接近时更容易优化”的核心结论。

In [ ]:
class R2Plus1DNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = R2Plus1DConv(1, 16)
        self.block2 = R2Plus1DConv(16, 32)
        self.block3 = R2Plus1DConv(32, 64)
        self.pool = nn.MaxPool3d(2)
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(64, NUM_CLASSES)

    def forward(self, x):
        x = self.pool(F.relu(self.block1(x)))   # (B, 1, 8, 16, 16) -> (B, 16, 4, 8, 8)
        x = self.pool(F.relu(self.block2(x)))   # (B, 16, 4, 8, 8) -> (B, 32, 2, 4, 4)
        x = self.gap(F.relu(self.block3(x)))    # (B, 32, 2, 4, 4) -> (B, 64, 1, 1, 1)
        x = x.flatten(1)                        # (B, 64)
        return self.fc(x)                       # (B, 3)


class R3DNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = R3DConv(1, 16)
        self.block2 = R3DConv(16, 32)
        self.block3 = R3DConv(32, 64)
        self.pool = nn.MaxPool3d(2)
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(64, NUM_CLASSES)

    def forward(self, x):
        x = self.pool(F.relu(self.block1(x)))   # (B, 1, 8, 16, 16) -> (B, 16, 4, 8, 8)
        x = self.pool(F.relu(self.block2(x)))   # (B, 16, 4, 8, 8) -> (B, 32, 2, 4, 4)
        x = self.gap(F.relu(self.block3(x)))    # (B, 32, 2, 4, 4) -> (B, 64, 1, 1, 1)
        x = x.flatten(1)                        # (B, 64)
        return self.fc(x)                       # (B, 3)


model_r21d = R2Plus1DNet()
model_r3d = R3DNet()
print(f'R(2+1)D params: {count_params(model_r21d):,}')
print(f'R3D params    : {count_params(model_r3d):,}')

## 七、训练阶段 vs 推理阶段

### 训练阶段
训练时我们会：
- 调用 `model.train()`
- 保留梯度，用于反向传播
- 让 `BatchNorm3d` 使用当前 batch 的统计量

### 推理阶段
推理时我们会：
- 调用 `model.eval()`
- 用 `torch.no_grad()` 关闭梯度
- 让 `BatchNorm3d` 使用累计统计量

> 对 CNN 视频模型来说，这里的 train/infer 差异主要不是“解码策略”，而是 **归一化层行为 + 是否保留梯度**。

In [ ]:
print('=== Train learning path: R(2+1)D ===')
losses_r21d = train_model(
    model_r21d,
    learning_train_loader,
    epochs=LEARNING_EPOCHS,
    lr=LEARNING_LR,
)
acc_r21d = evaluate_model(model_r21d, test_loader)
print(f'R(2+1)D test accuracy: {acc_r21d:.2%}')

print('\n=== Train learning path baseline: R3D ===')
losses_r3d = train_model(
    model_r3d,
    learning_train_loader,
    epochs=LEARNING_EPOCHS,
    lr=LEARNING_LR,
)
acc_r3d = evaluate_model(model_r3d, test_loader)
print(f'R3D test accuracy: {acc_r3d:.2%}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses_r21d, label='R(2+1)D', linewidth=2)
ax.plot(losses_r3d, label='R3D', linewidth=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Learning Path: R(2+1)D vs R3D')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 八、工程路径：使用 torchvision builder

官方文档给出的 `torchvision.models.video.r2plus1d_18` 签名是：

```python
r2plus1d_18(*, weights: Optional[R2Plus1D_18_Weights] = None, progress: bool = True, **kwargs) -> VideoResNet
```

几点要点：
- 若 `weights=None`，表示不加载预训练权重
- 预训练权重默认别名是 `R2Plus1D_18_Weights.DEFAULT = KINETICS400_V1`
- 预训练推理时，官方 transform 会把输入整理为 `(..., C, T, H, W)`，并使用 `resize=[128,171]`、`crop=[112,112]`、指定 mean/std

本 Notebook 为了满足“**同一任务 + 不依赖外网下载 + CPU/GPU 都可运行**”，工程路径采用：
- **同样的 synthetic video 数据**
- **builder 直接建模**
- **不加载预训练权重**
- **只训练极小子集 + 极少 epoch**，重点展示高层 API 的工程表达

### 黑盒拆解：builder 背后做了什么？
从模块思想看，工程路径仍然在做同一件事：
1. 期望输入是视频张量 `(B, C, T, H, W)`
2. stem 先做时空卷积特征提取
3. 后续残差层继续堆叠 `(2+1)D` block
4. 最后通过池化 + 全连接输出类别

也就是说：**学习路径在手写 block，工程路径在复用成熟 block 组合。**

In [ ]:
engineering_model = r2plus1d_18(weights=None, num_classes=NUM_CLASSES)
print(engineering_model)
print(f'Engineering model params: {count_params(engineering_model):,}')

sample_rgb = adapt_to_rgb(sample_videos[:2])
print(f'Builder input shape: {tuple(sample_rgb.shape)}')
with torch.no_grad():
    builder_logits = engineering_model(sample_rgb)
print(f'Builder output shape: {tuple(builder_logits.shape)}')

## 九、工程路径训练、推理与对照

### Component correspondence

| 学习路径实现 | 工程路径内部对应 | 说明 |
|---|---|---|
| `R2Plus1DConv.spatial` | `r2plus1d_18` block 里的 spatial conv | 都是在时间维保持不变时先提取空间特征 |
| `R2Plus1DConv.temporal` | builder block 里的 temporal conv | 都是在空间特征之后沿时间维融合 |
| 手写 `R2Plus1DNet` | `VideoResNet` | 一个是教学版，一个是工业版 |
| `adapt_to_rgb()` | 官方预处理里对输入通道/布局的要求 | 工业模型对输入格式更严格 |
| `evaluate_model(..., no_grad)` | `model.eval()` + 前向推理 | 推理阶段都关闭梯度 |

### 工程路径的训练与推理差异
- 训练：`model.train()`，更新参数，BatchNorm 用 batch 统计
- 推理：`model.eval()`，不更新参数，BatchNorm 用累计统计
- 与学习路径底层算法一致，只是封装程度更高

In [ ]:
print('=== Train engineering path: torchvision r2plus1d_18 ===')
engineering_model = r2plus1d_18(weights=None, num_classes=NUM_CLASSES)
losses_engineering = train_model(
    engineering_model,
    engineering_train_loader,
    epochs=ENGINEERING_EPOCHS,
    lr=ENGINEERING_LR,
    input_adapter=adapt_to_rgb,
)
acc_engineering = evaluate_model(
    engineering_model,
    test_loader,
    input_adapter=adapt_to_rgb,
)
print(f'Engineering path test accuracy: {acc_engineering:.2%}')

infer_videos, infer_labels = next(iter(test_loader))
logits_learning, preds_learning = predict_batch(model_r21d, infer_videos[:6])
logits_engineering, preds_engineering = predict_batch(
    engineering_model,
    infer_videos[:6],
    input_adapter=adapt_to_rgb,
)

print('Ground truth        :', infer_labels[:6].tolist())
print('Learning preds      :', preds_learning.tolist())
print('Engineering preds   :', preds_engineering.tolist())

## 十、学习路径 vs 工程路径对比

| 对比维度 | 学习路径 | 工程路径 |
|---------|---------|---------|
| 教育价值 | 能看到 `M_i` 公式、shape 变化、R3D 对照关系 | 能理解工业 API 如何把复杂结构封装成一行 builder |
| 代码量 | 较多，需要自己写 block、网络、训练逻辑 | 很少，核心建模可压缩到一行 builder |
| 灵活性 | 高，可以改任意 block、通道数、训练流程 | 中等，受限于库提供的接口与默认结构 |
| 稳定性 | 一般，手写实现更容易出 shape bug | 高，成熟实现经过大量验证 |
| 工业适配度 | 适合研究原型、原理论证、教学演示 | 更适合复用、实验迭代、工程落地 |
| 适用场景 | 面试讲原理；论文复现；做结构改造 | 快速 baseline；项目试验；接入已有训练框架 |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(losses_r21d, label='Learning: R(2+1)D', linewidth=2)
axes[0].plot(losses_r3d, label='Learning: R3D', linestyle='--')
axes[0].plot(
    np.linspace(0, len(losses_r21d) - 1, len(losses_engineering)),
    losses_engineering,
    label='Engineering: r2plus1d_18',
    marker='o'
)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Loss Curves Across Paths')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

names = ['Learning\nR(2+1)D', 'Learning\nR3D', 'Engineering\nr2plus1d_18']
accs = [acc_r21d, acc_r3d, acc_engineering]
params = [count_params(model_r21d), count_params(model_r3d), count_params(engineering_model)]
bars = axes[1].bar(names, accs, color=['#2ecc71', '#e74c3c', '#3498db'])
for bar, acc, p in zip(bars, accs, params):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.015,
        f'{acc:.1%}\n({p/1e6:.2f}M)',
        ha='center',
        fontsize=8,
    )
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Accuracy and Parameter Scale')
axes[1].set_ylim(0, 1.15)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('Path summary:')
print(f'- Learning R(2+1)D accuracy : {acc_r21d:.2%}')
print(f'- Learning R3D accuracy      : {acc_r3d:.2%}')
print(f'- Engineering accuracy       : {acc_engineering:.2%}')

## 十一、训练与推理差异总表

| 阶段 | 学习路径行为 | 工程路径行为 |
|------|------------|------------|
| 训练 | `model.train()`；手写训练循环；参数更新；`BatchNorm3d` 使用当前 batch 统计 | `train_model(...)` + `r2plus1d_18`；高层结构被封装，但底层仍是同样的梯度优化 |
| 推理 | `predict_batch()` + `model.eval()` + `torch.no_grad()` | `predict_batch(..., input_adapter=adapt_to_rgb)` + `model.eval()` |
| 输入格式 | 灰度 `(B,1,T,H,W)` | 复制为 RGB `(B,3,T,H,W)` 以匹配 builder 预期 |
| 算法本质 | 手写 `(2+1)D` block 前向传播 | 调用成熟实现的前向传播 |

> 结论：两条路径在 **训练/推理机制** 上没有本质矛盾，差别主要在 **抽象层级** 和 **输入规范**。

## 十二、结果解读与边界

### 学习路径的收益与边界
- **收益**：能精确解释 `M_i` 如何匹配参数量，能看清空间卷积、时间卷积、ReLU 各自负责什么
- **边界**：这里是 toy 数据，结论只能说明“分解思路可运行、且更容易优化的趋势可被小规模观察到”，不能替代大规模真实视频实验

### 工程路径的收益与边界
- **收益**：几乎一行 builder 就能拿到标准架构，适合快速接入训练框架和项目代码
- **边界**：黑盒程度更高，不容易直接看到每个模块的设计动机；真实预训练场景通常还涉及官方 transforms、视频解码与更高分辨率输入

### 实践建议
- 如果目标是 **面试 / 论文理解 / 模型改造**：先走学习路径
- 如果目标是 **项目 baseline / 快速复现 / 工程接入**：优先走工程路径
- 最理想的方式是：**用学习路径建立认知，用工程路径完成落地**

## 十三、面试与项目表达

### 高频面试题

**Q1：R(2+1)D 的核心创新到底是什么？**

- 把一个 `t × d × d` 的 3D 卷积拆成 **空间卷积 `(1,d,d)` + 时间卷积 `(t,1,1)`**
- 在两者之间插入一次 ReLU，带来更强非线性
- 在参数量近似不变时，优化往往更容易

**Q2：为什么 R(2+1)D 往往比 R3D 更容易训练？**

- 因为空间模式和时间模式被拆开学习，优化目标更清晰
- 中间额外多了一次非线性
- 论文中给出的证据是：在相近容量下，R(2+1)D 的训练误差更低

**Q3：`M_i` 公式的意义是什么？**

- 不是为了“神奇提分”，而是为了让 `(2+1)D` 分解与原始 3D 卷积在参数规模上尽量公平
- 这样对比 R3D 与 R(2+1)D 时，更容易把差异归因到结构而不是参数量

**Q4：R(2+1)D 和 I3D / C3D 的关系是什么？**

- **C3D**：早期全 3D 卷积视频模型
- **I3D**：把 2D 卷积 inflate 成 3D 卷积
- **R3D**：ResNet 风格全 3D 卷积
- **R(2+1)D**：在 ResNet 风格结构里把 3D 卷积进一步分解

**Q5：工程里什么时候优先用 builder，而不是手写实现？**

- 当你需要快速做 baseline、接入训练框架、减少 shape bug 时
- 当重点是项目落地而不是研究某个 block 的内部机制时

**Q6：为什么工程路径里要把灰度复制成 3 通道？**

- 因为 `torchvision` 视频模型默认按 RGB 视频设计
- builder 预期输入通道数为 3，所以 toy 灰度视频需要适配输入格式

**Q7：如果面试官追问“训练阶段和推理阶段有什么差别”，应该怎么答？**

- 对这类 CNN 视频模型，核心差别通常不是自回归解码
- 真正要答的是：`model.train()` / `model.eval()`、BatchNorm 的统计方式、以及是否保留梯度

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|------|-----------|
| 1 | R(2+1)D 在做什么？ | 把 3D 卷积拆成空间 2D + 时间 1D |
| 2 | 为什么它更容易训练？ | 空间与时间解耦 + 多一次 ReLU |
| 3 | `M_i` 是干什么的？ | 让分解前后参数量尽量匹配，保证公平比较 |
| 4 | 和 R3D 最大区别？ | R3D 联合卷积，R(2+1)D 分步卷积 |
| 5 | 工程里怎么快速用？ | 直接用 `torchvision.models.video.r2plus1d_18` |
| 6 | 训练/推理差别答什么？ | `train/eval` 模式、BatchNorm、梯度开关 |

### 项目表达

> 如果面试官问“你做过什么相关项目”，可以这样组织：

- **学习深度**：我从零实现了 R(2+1)D 的空间卷积、时间卷积和 `M_i` 参数匹配公式，并手写了与 R3D 的对照实验
- **工程能力**：我用 `torchvision` 的 `r2plus1d_18` builder 在同一任务上复现了高层工程实现，理解了输入格式和接口约束
- **对比思考**：我能解释为什么学习路径适合理解原理，工程路径适合快速落地，也能说清楚两者在训练/推理阶段的共同点与差别

### 延伸阅读与对比

| 对比维度 | R(2+1)D | R3D | SlowFast |
|---------|---------|-----|----------|
| 核心思想 | 3D 分解为空间 + 时间 | 全 3D 卷积 | 双路径建模慢语义 + 快运动 |
| 优势 | 易优化、表达更强 | 结构直接、概念简单 | 时序建模能力更强 |
| 代价 | 结构稍复杂 | 优化相对更难 | 工程复杂度更高 |
| 适用场景 | 兼顾原理与工程 | 对照基线 | 更高性能视频理解 |

### 进阶探索方向

- 把 toy 数据替换成真实小型视频数据集，观察结论是否仍成立
- 对比 `r3d_18` 与 `r2plus1d_18` 的 builder 级实验
- 进一步阅读 SlowFast、X3D，理解 R(2+1)D 思想如何影响后续视频模型

In [ ]:
# Optional appendix: visualize intermediate activations
with torch.no_grad():
    model_r21d.eval()
    feat1 = model_r21d.block1(infer_videos[:1].to(device)).cpu()   # (1, 16, T, H, W)

fig, axes = plt.subplots(1, 4, figsize=(10, 2.5))
for i, frame_id in enumerate([0, 2, 4, 6]):
    axes[i].imshow(feat1[0, 0, frame_id], cmap='viridis')
    axes[i].set_title(f'Act map t={frame_id}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()